# UFC Fighter Rating

This notebook runs the whole pipeline and shows its two products:

1. **Fight-outcome models**: how well can the result of a UFC fight be predicted from what was known before it?
2. **Division rankings**: the active fighters of each division, ranked by three independent methods.

Each step writes its output to `data/processed/`. The three analysis notebooks read those files, so run this one first:

| Notebook | Content |
|---|---|
| [01_exploration](notebooks/01_exploration.ipynb) | The data: coverage, how fights end, the corner artefact, the betting market |
| [02_feature_engineering](notebooks/02_feature_engineering.ipynb) | How every feature is built without leaking the future, and how much signal each one carries |
| [03_models_and_rankings](notebooks/03_models_and_rankings.ipynb) | Model diagnostics, and a critical look at the rankings |

The same pipeline runs from the command line with `python -m ufc_rating.pipeline`.

## 0. Configuration

In [1]:
# Data
REFRESH_DATA = False      # True: download the latest Kaggle snapshots first (needs Kaggle API credentials)
SCRAPE_UFCSTATS = False   # also try ufcstats.com for newer events (it answers bots with a challenge page)

# Weighted ranking: weight of each career statistic (must sum to 1)
WEIGHTS = {
    'win_rate':    0.20,   # wins / decided fights
    'finish_rate': 0.15,   # wins by KO/TKO or submission
    'slpm':        0.15,   # significant strikes landed per minute
    'sig_acc':     0.10,   # significant strike accuracy
    'td_per15':    0.10,   # takedowns per 15 minutes
    'td_acc':      0.10,   # takedown accuracy
    'ctrl_pct':    0.10,   # share of fight time in control
    'kd_per15':    0.05,   # knockdowns per 15 minutes
    'sub_per15':   0.05,   # submission attempts per 15 minutes
}

# Who gets ranked
ACTIVE_DAYS = 730   # fought in the last two years
MIN_FIGHTS = 5      # at least five UFC fights
TOP_N = 10          # fighters shown per division

In [2]:
import pandas as pd
from IPython.display import Markdown, display

from ufc_rating.config import DIVISIONS
from ufc_rating.models.training import format_scores
from ufc_rating.pipeline import build_features, fit_models, rank_fighters, update_data

pd.set_option('display.width', 140)

## 1. Data

- **Fights**: a CC0 Kaggle mirror of [ufcstats.com](http://ufcstats.com), the official UFC statistics site: results, fight totals (strikes, takedowns, control time...) and fighter profiles. Only UFC events are kept.
- **Betting odds and official ranks** at fight time, from 2010 onwards (Kaggle, CC BY 4.0), matched to the fights on names and date.
- **Our own scraper** (`src/ufc_rating/ingest/ufcstats.py`) can add events more recent than the mirror. Since 2026 ufcstats.com answers automated clients with a JavaScript challenge; the scraper detects it and stops instead of trying to get around it.

In [3]:
master = update_data(refresh=REFRESH_DATA, scrape=SCRAPE_UFCSTATS)

Master table: 8,828 UFC fights, 1993-11-12 to 2026-08-08


## 2. Features

Every decided fight becomes a comparison between two fighters **A** and **B**, drawn at random from the two corners, described by the difference of their career statistics, physical attributes and Elo rating **before** the fight. Draws, no contests and fights involving a UFC debutant (no history to learn from) are left out.

Why the corners have to be randomised, and how each feature avoids leaking the future: [02_feature_engineering](notebooks/02_feature_engineering.ipynb).

In [4]:
elo_history, matchups, profiles = build_features(master)

Matchups: 6,465 fights with two experienced fighters, 24 stat features (+ odds)


## 3. Models

The fights are split in time: the oldest 70% for training, the next 15% for validation, the most recent 15% for the final test. Four models (logistic regression, SVM, random forest, XGBoost) are tuned by time-ordered cross-validation on the training period, in two versions:

- **stats**: fighter data only. This is the model used for the rankings.
- **stats + odds**: the same inputs plus the bookmakers' implied probability.

Validation picks the stats model used for the rankings; the test period is scored once, at the end. For the rankings, which describe the fighters of today, the selected model is then refitted on every fight.

In [5]:
fitted = fit_models(matchups)
results = fitted['results']

pd.DataFrame(results['split'], index=['from', 'to', 'fights']).T

Stats-only models:


  LogReg        CV log-loss 0.6635  {'C': 0.01}


  SVM           CV log-loss 0.6643  {'C': 0.01, 'kernel': 'linear'}


  RandomForest  CV log-loss 0.6700  {'max_depth': 8, 'min_samples_leaf': 10}


  XGBoost       CV log-loss 0.6711  {'learning_rate': 0.02, 'max_depth': 2}
Stats + odds models:
  LogReg        CV log-loss 0.6321  {'C': 0.01}


  SVM           CV log-loss 0.6350  {'C': 0.01, 'kernel': 'linear'}


  RandomForest  CV log-loss 0.6495  {'max_depth': 8, 'min_samples_leaf': 10}


  XGBoost       CV log-loss 0.6437  {'learning_rate': 0.02, 'max_depth': 2}


,from,to,fights
train,1994-03-11 00:00:00,2021-11-20 00:00:00,4515
validation,2021-12-04 00:00:00,2024-04-13 00:00:00,972
test,2024-04-27 00:00:00,2026-08-08 00:00:00,978


**Test period, all fights.** Accuracy is the share of winners correctly predicted (50% is a coin flip, since A is drawn at random). AUC measures how well the predicted probabilities rank winners above losers (0.5 = chance). Log loss and Brier score measure the quality of the probabilities themselves (lower is better). *Elo only* predicts with the pre-fight Elo ratings alone.

In [6]:
format_scores(results['test_stats'])

,Fights,Accuracy,AUC,Log loss,Brier
LogReg,978,66.0%,0.713,0.628,0.219
SVM,978,65.8%,0.713,0.627,0.218
RandomForest,978,64.5%,0.697,0.643,0.226
XGBoost,978,62.9%,0.699,0.638,0.224
Elo only,978,54.4%,0.575,0.684,0.246


**Against the betting market.** Only the test fights with odds, so every line is scored on exactly the same fights.

In [7]:
format_scores(results['test_market'])

,Fights,Accuracy,AUC,Log loss,Brier
Betting favourite (market),725,70.2%,0.766,0.578,0.197
Elo only,725,53.9%,0.570,0.685,0.246
LogReg (stats),725,66.9%,0.723,0.623,0.217
SVM (stats),725,66.6%,0.722,0.623,0.216
RandomForest (stats),725,65.1%,0.702,0.641,0.225
XGBoost (stats),725,63.7%,0.708,0.635,0.222
LogReg (stats + odds),725,70.3%,0.768,0.577,0.196
SVM (stats + odds),725,69.4%,0.767,0.578,0.197
RandomForest (stats + odds),725,70.5%,0.761,0.594,0.203
XGBoost (stats + odds),725,70.2%,0.764,0.583,0.199


Reading these two tables:

- Fighter statistics alone predict the winner clearly better than chance and better than Elo alone.
- The betting market stays the reference: bookmakers see everything the statistics see, plus injuries, camps, weight cuts and style match-ups.
- Adding the statistics to the odds brings the models level with the market, not above it: the public statistics carry no information the market has not already priced in.

Calibration, confidence and what each model relies on: [03_models_and_rankings](notebooks/03_models_and_rankings.ipynb).

## 4. Rankings

Active fighters (a fight in the last `ACTIVE_DAYS` days, at least `MIN_FIGHTS` UFC fights) are ranked in their most recent division by three methods:

- **Elo**: dynamic rating updated after every fight since 1993 (win = 1, draw = 0.5, K = 32). It rewards *who* you beat.
- **Weighted**: career statistics turned into percentiles within the division and combined with the `WEIGHTS` above. It rewards *how* you fight.
- **Model**: a virtual round-robin tournament. The selected stats model, refitted on all fights, predicts every possible match-up in the division; the score is the fighter's average win probability.

Fighters are sorted by the **consensus**, the mean of the three ranks.

In [8]:
as_of = master['date'].max()
best = results['best_stats_model']
rankings = rank_fighters(profiles, fitted['ranking_model'], as_of, WEIGHTS,
                         active_days=ACTIVE_DAYS, min_fights=MIN_FIGHTS)
print(f'Rankings as of {as_of.date()}, round-robin model: {best}')

Rankings as of 2026-08-08, round-robin model: LogReg


In [9]:
columns = ['Fighter', 'UFC record', 'Last fight', 'Elo rank', 'Weighted rank', 'Model rank', 'Consensus']
for division in DIVISIONS:
    table = rankings[rankings['division'] == division]
    if table.empty:
        continue
    display(Markdown(f'### {division} ({len(table)} active fighters)'))
    display(table.set_index('rank')[columns].head(TOP_N))

### Flyweight (35 active fighters)

,Fighter,UFC record,Last fight,Elo rank,Weighted rank,Model rank,Consensus
rank,,,,,,,
1,Joshua Van,10-1,2026-05-09,2,3,1,2.0
2,Alexandre Pantoja,14-4,2025-12-06,1,1,7,3.0
3,Tatsuro Taira,8-2,2026-05-09,6,2,2,3.3
4,Asu Almabayev,7-1,2026-06-27,5,4,4,4.3
5,Manel Kape,8-3,2026-06-20,4,7,3,4.7
6,Kyoji Horiguchi,9-2,2026-06-20,3,8,9,6.7
7,Andre Lima,4-1,2026-06-20,12,6,5,7.7
8,Brandon Moreno,11-7-2,2026-02-28,7,14,8,9.7
9,Tagir Ulanbekov,6-2,2025-11-22,9,9,13,10.3


### Bantamweight (51 active fighters)

,Fighter,UFC record,Last fight,Elo rank,Weighted rank,Model rank,Consensus
rank,,,,,,,
1,Umar Nurmagomedov,8-1,2026-01-24,5,3,2,3.3
2,Sean O'Malley,12-3,2026-06-14,2,5,3,3.3
3,Mario Bautista,12-3,2026-07-11,4,2,9,5.0
4,Merab Dvalishvili,14-3,2025-12-06,1,14,4,6.3
5,Petr Yan,12-4,2025-12-06,3,7,10,6.7
6,Farid Basharat,7-0,2026-07-11,7,12,5,8.0
7,Raul Rosas Jr.,6-1,2026-03-07,17,8,1,8.7
8,Payton Talbott,5-1,2025-12-06,19,1,7,9.0
9,Bryce Mitchell,10-3,2026-06-06,9,11,8,9.3


### Featherweight (57 active fighters)

...

,Fighter,UFC record,Last fight,Elo rank,Weighted rank,Model rank,Consensus
rank,,,,,,,
1,Tatiana Suarez,9-1,2026-04-11,1,2,2,1.7
2,Fatima Kline,4-1,2026-07-18,8,1,1,3.3
3,Gillian Robertson,14-6,2026-03-14,2,5,5,4.0
4,Denise Gomes,6-2,2025-11-08,6,3,4,4.3
5,Virna Jandiroba,9-4,2026-04-04,3,9,11,7.7
6,Jaqueline Amorim,5-2,2026-05-30,11,7,6,8.0
7,Loopy Godinez,9-6,2026-04-11,13,6,8,9.0
8,Iasmin Lucindo,5-2,2025-08-09,9,16,3,9.3
9,Piera Rodriguez,5-2,2026-03-14,12,8,10,10.0


### Women's Flyweight (30 active fighters)

,Fighter,UFC record,Last fight,Elo rank,Weighted rank,Model rank,Consensus
rank,,,,,,,
1,Valentina Shevchenko,15-3-1,2025-11-15,1,4,3,2.7
2,Erin Blanchfield,8-1,2025-11-15,5,3,2,3.3
3,Zhang Weili,10-3,2025-11-15,3,1,10,4.7
4,Natalia Silva,8-0,2026-01-24,2,12,1,5.0
5,Maycee Barber,10-3,2026-03-28,6,5,4,5.0
6,Casey O'Neill,6-2,2026-03-28,13,2,5,6.7
7,Manon Fiorot,8-1,2025-10-18,4,8,8,6.7
8,Wang Cong,5-1,2026-07-11,10,7,7,8.0
9,Jasmine Jasudavicius,9-3,2026-04-18,7,6,15,9.3


### Women's Bantamweight (22 active fighters)

,Fighter,UFC record,Last fight,Elo rank,Weighted rank,Model rank,Consensus
rank,,,,,,,
1,Luana Santos,6-1,2026-06-20,4,1,1,2.0
2,Ailin Perez,6-1,2026-02-28,3,2,2,2.3
3,Joselyne Edwards,9-4,2026-04-25,5,3,4,4.0
4,Norma Dumont,9-3,2026-04-25,2,7,5,4.7
5,Raquel Pennington,13-6,2024-10-05,1,10,8,6.3
6,Julianna Pena,8-4,2025-06-07,7,4,9,6.7
7,Karol Rosa,8-5,2026-06-20,11,5,6,7.3
8,Macy Chiasson,8-6,2026-02-28,13,6,7,8.7
9,Jacqueline Cavalcanti,5-1,2026-05-16,8,16,3,9.0


How far the three methods agree, how they compare with the official UFC rankings, and the all-time Elo table: [03_models_and_rankings](notebooks/03_models_and_rankings.ipynb).